In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Helper Function for Text Cleaning:

Implement a Helper Function as per Text Preprocessing Notebook and Complete the following pipeline.

# Build a Text Cleaning Pipeline

In [ ]:
def text_cleaning_pipeline(dataset, rule = "lemmatize"):
  """
  This...
  """
  # Convert the input to small/lower order.
  data =
  # Remove URLs
  data =
  # Remove emojis
  data =
  # Remove all other unwanted characters.
  data =
  # Create tokens.
  tokens = data.split()
  # Remove stopwords:
  tokens =
  if rule == "lemmatize":
    tokens =
  elif rule == "stem":
    tokens =
  else:
    print("Pick between lemmatize or stem")


  return " ".join(tokens)


# Text Classification using Machine Learning Models


### 📝 Instructions: Trump Tweet Sentiment Classification

1. **Load the Dataset**  
   Load the dataset named `"trump_tweet_sentiment_analysis.csv"` using `pandas`. Ensure the dataset contains at least two columns: `"text"` and `"label"`.

2. **Text Cleaning and Tokenization**  
   Apply a text preprocessing pipeline to the `"text"` column. This should include:
   - Lowercasing the text  
   - Removing URLs, mentions, punctuation, and special characters  
   - Removing stopwords  
   - Tokenization (optional: stemming or lemmatization)
   - "Complete the above function"

3. **Train-Test Split**  
   Split the cleaned and tokenized dataset into **training** and **testing** sets using `train_test_split` from `sklearn.model_selection`.

4. **TF-IDF Vectorization**  
   Import and use the `TfidfVectorizer` from `sklearn.feature_extraction.text` to transform the training and testing texts into numerical feature vectors.

5. **Model Training and Evaluation**  
   Import **Logistic Regression** (or any machine learning model of your choice) from `sklearn.linear_model`. Train it on the TF-IDF-embedded training data, then evaluate it using the test set.  
   - Print the **classification report** using `classification_report` from `sklearn.metrics`.


In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/workshop8/trum_tweet_sentiment_analysis.csv')
display(df.head())

,text,Sentiment
0,RT @JohnLeguizamo: #trump not draining swamp b...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0
2,Trump protests: LGBTQ rally in New York https:...,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0


In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

# Download NLTK data (run once)
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')
try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

def text_cleaning_pipeline(dataset, rule = "lemmatize"):
  """
  This function cleans text data by performing a series of preprocessing steps.
  Steps include: lowercasing, removing URLs, emojis, mentions, punctuation,
  stopwords, and optionally lemmatizing or stemming.
  """
  # Convert the input to small/lower order.
  data = dataset.lower()

  # Remove URLs
  data = re.sub(r"http\S+|www\S+|https\S+", '', data, flags=re.MULTILINE)

  # Remove mentions (@username)
  data = re.sub(r'@\w+', '', data)

  # Remove emojis (simple way: remove non-ascii characters)
  # This also effectively removes many special characters that are not part of common text.
  data = data.encode('ascii', 'ignore').decode('ascii')

  # Remove all other unwanted characters (punctuation and special characters)
  # Keep only alphanumeric and spaces
  data = re.sub(r'[^a-z0-9\s]', '', data)

  # Remove extra whitespaces
  data = re.sub(r'\s+', ' ', data).strip()

  # Create tokens.
  tokens = data.split()

  # Remove stopwords:
  tokens = [word for word in tokens if word not in stop_words]

  if rule == "lemmatize":
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
  elif rule == "stem":
    tokens = [stemmer.stem(word) for token in tokens]
  else:
    print("Pick between lemmatize or stem")

  return " ".join(tokens)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [ ]:
df['cleaned_text'] = df['text'].apply(text_cleaning_pipeline)
display(df.head())

,text,Sentiment,cleaned_text
0,RT @JohnLeguizamo: #trump not draining swamp b...,0,rt trump draining swamp taxpayer dollar trip a...
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0,icymi hacker rig fm radio station play antitru...
2,Trump protests: LGBTQ rally in New York https:...,1,trump protest lgbtq rally new york bbcworld via
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0,hi im pier morgan david beckham awful donald t...
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0,rt tech firm suing buzzfeed publishing unverif...


In [ ]:
from sklearn.model_selection import train_test_split

X = df['cleaned_text']
y = df['Sentiment'] # Assuming 'Sentiment' is the label column based on 'trump_tweet_sentiment_analysis.csv'

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data samples: {X_train.shape[0]}")
print(f"Testing data samples: {X_test.shape[0]}")

Training data samples: 1480098
Testing data samples: 370025


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_features=5000) # You can adjust max_features
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

print(f"TF-IDF transformed training data shape: {X_train_tfidf.shape}")
print(f"TF-IDF transformed testing data shape: {X_test_tfidf.shape}")

TF-IDF transformed training data shape: (1480098, 5000)
TF-IDF transformed testing data shape: (370025, 5000)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Initialize and train the Logistic Regression model
logistic_model = LogisticRegression(max_iter=1000, random_state=42) # Increased max_iter for convergence on large dataset
logistic_model.fit(X_train_tfidf, y_train)

# Make predictions on the test set
y_pred = logistic_model.predict(X_test_tfidf)

# Print the classification report
print("Classification Report for Logistic Regression:")
print(classification_report(y_test, y_pred))

Classification Report for Logistic Regression:
              precision    recall  f1-score   support

           0       0.93      0.96      0.94    248563
           1       0.90      0.86      0.88    121462

    accuracy                           0.92    370025
   macro avg       0.92      0.91      0.91    370025
weighted avg       0.92      0.92      0.92    370025

